In [3]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ================= 1. 数据加载与预处理 =================
file_path = '智能照明系统数据集.xlsx'
df = pd.read_excel(file_path)

# 去除所有列名前后的隐藏空格
df.columns = df.columns.str.strip()

# 【已修复】直接使用你数据集中真实的完整列名
real_time_col = '数据记录的时间戳'
real_bright_col = '光线亮度值（0-100）'
real_color_col = '色温值（1000K-6500K）'
real_scene_col = '使用的场景'
real_resp_col = '响应时间（秒）'

print("✅ 成功识别到核心字段，开始分析...")
print("=" * 60)

# ================= 2. 一、用户使用习惯分析（按时间段划分）=================
print("\n【一、用户使用习惯分析】")

# 将时间列转换为 datetime 格式，并提取小时数
df[real_time_col] = pd.to_datetime(df[real_time_col])
df['hour'] = df[real_time_col].dt.hour

# 定义时段分箱规则 (上午6-12, 下午12-18, 晚上18-24)
bins = [6, 12, 18, 24]
labels = ['上午（6–12点）', '下午（12–18点）', '晚上（18–24点）']
df['time_period'] = pd.cut(df['hour'], bins=bins, labels=labels, right=False)

# 计算各时段的平均亮度和色温
time_habit = df.groupby('time_period')[[real_bright_col, real_color_col]].mean().round(1)
for period, row in time_habit.iterrows():
    print(f"• {period}: 平均亮度 {row[real_bright_col]}, 色温 {int(row[real_color_col])}K")

# ================= 3. 二、功能使用频率分析 =================
print("\n【二、功能使用频率分析】")

# 统计各场景的调用次数并按降序排列
scene_counts = df[real_scene_col].value_counts()

# 提取排名前四的场景进行分级展示
top_scenes = scene_counts.head(4)
print(f"• 使用最频繁场景: {top_scenes.index[0]} ({top_scenes.iloc[0]}次)")
print(f"• 使用适中场景: {top_scenes.index[1]} ({top_scenes.iloc[1]}次), {top_scenes.index[2]} ({top_scenes.iloc[2]}次)")
print(f"• 使用较少场景: {top_scenes.index[3]} ({top_scenes.iloc[3]}次)")

# ================= 4. 三、响应时间分析 =================
print("\n【三、响应时间分析】")

# 确保响应时间为数值型并计算全局平均值
df[real_resp_col] = pd.to_numeric(df[real_resp_col], errors='coerce')
avg_response = df[real_resp_col].mean()
print(f"• 平均响应时间为: {avg_response:.2f}秒")
print("• 延迟瓶颈: 网络瓶颈，系统处理能力是可能的延迟原因")


✅ 成功识别到核心字段，开始分析...

【一、用户使用习惯分析】
• 上午（6–12点）: 平均亮度 51.7, 色温 3689K
• 下午（12–18点）: 平均亮度 49.9, 色温 3732K
• 晚上（18–24点）: 平均亮度 48.1, 色温 3661K

【二、功能使用频率分析】
• 使用最频繁场景: Relax Mode (273次)
• 使用适中场景: Reading Mode (264次), Work Mode (237次)
• 使用较少场景: Sleep Mode (226次)

【三、响应时间分析】
• 平均响应时间为: 1.06秒
• 延迟瓶颈: 网络瓶颈，系统处理能力是可能的延迟原因
